# POC for distributed data alignment


## Preparation of the dataset

We use Adult as typical benchmark dataset.

### Dataset loading

We are only interested in the features

In [ ]:
import os
import sys
from pathlib import Path

base_path = Path(os.path.abspath("")).parent.joinpath("src")
sys.path.append(base_path.as_posix())

In [ ]:
# !uv pip install kagglehub

In [ ]:
from pathlib import Path

import kagglehub
import pandas as pd

dataset_names = {
    # "kimjihoo/coronavirusdataset": "Case.csv",
    # "sudalairajkumar/covid19-in-india": "covid_19_india.csv",
    "georgesaavedra/covid19-dataset": "owid-covid-data.csv",
    "hendratno/covid19-indonesia": "covid_19_indonesia_time_series_all.csv",
}
datasets = {}
for dataset_name, file_name in dataset_names.items():
    path = kagglehub.dataset_download(dataset_name)
    csv_path = Path(path).joinpath(file_name).as_posix()
    print(csv_path)
    datasets[dataset_name] = pd.read_csv(csv_path)

In [ ]:
# output_file_path = "./data/covid/"

### Parties creation

Let's split the dataset and create different parties relying on the same schema (to begin with)

In [ ]:
A, B = datasets["georgesaavedra/covid19-dataset"], datasets["hendratno/covid19-indonesia"]

And let us augment the dataset adding a date column, for no particular reason other than it is useful

In [ ]:
import datetime
import random


def generate_dates(many: int, format: str = r"%Y-%m-%d") -> list[str]:
    return [
        datetime.date(
            int(1980 + 50 * random.random()), int(6 + random.random() * 6), int(15 + 10 * random.random())
        ).strftime(format)
        for _ in range(many)
    ]

In [ ]:
A.head()

In [ ]:
B.head()

In [ ]:
from privfusion.dataset_analyzer import DatasetAnalyzer

analyzer = DatasetAnalyzer()

In [ ]:
sem_info_a = analyzer.extract_information_from_dataset(A)
sem_info_a

In [ ]:
sem_info_b = analyzer.extract_information_from_dataset(B)
sem_info_b